[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/lens_camera/lens_camera.ipynb)

# A Zoom Camera with Depth of Field

This example photographs a grid of points at three depths through a two-lens zoom. A point at the focused depth images to a sharp dot; a point nearer or farther spreads its light over a blur disc, larger the further it is from focus and the wider the aperture. Moving the rear lens zooms, moving the sensor refocuses, and tilting the sensor brings a slanted plane into focus.

An ideal thin lens is a collineation of space, a map from points to points, `Point + origin * (home & Point) / focal`, and its action on lines is the join of the images of their points: a train of lenses composes either way. The rays a scene point sends through the aperture form a cone, a quadric, the aperture's ball pulled back through the projection from the point onto the aperture plane by its adjugate. The train carries that cone to the image cone, a pullback through the inverse of its point map, and the sensor cuts the image cone in the point's blur conic. Nothing asks where the point focuses: the cone's vertex is wherever the collineation puts it.

In the matrix notation of projective geometry the lens reads as a $4\times4$ homography $H$, the aperture cone as $Q' = P^{\mathsf T} Q P$ for the projection $P$, and the image cone as $H^{-\mathsf T} Q' H^{-1}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from numga.gatype.traits import Versor
from examples.util.animation import save_figure, show_animation
from examples.optics.lens_camera import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                                                  # Antivector-
Plane = ga.gatype.vector()                                                      # Vector+
SensorPoint = Point.derive.containing("x")                                      # Antivector-
SensorPlane = Plane.derive.within("y z w")                                      # Vector+
# Lines are joins of points.
Line = ga.gatype.antibivector()                                                 # Antibivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
# A collineation maps points to points.
PointMap = ga.gatype((Point, Point))                                            # Point- <- Point-
LineMap = ga.gatype((Line, Line))                                               # Line- <- Line-
# A primal quadric maps a point to its polar plane.
Quadric = ga.gatype((Plane, Point))                                             # Plane+ <- Point-
# A camera maps a scene point and a pupil point to a sensor point.
Camera = ga.gatype((Point, Point, Point))                                       # Point- <- (Point-, Point-)


def unit(points: Point) -> Point:
    """The representative of each point with positive unit weight; a unit point is a versor."""
    return (points / (mv.w & points)).normalized().with_traits(Versor)


# Every element is built in its home frame: centred on the origin, in the plane x == 0.
origin = unit(mv.w.dual())                                                      # [] Point-
home = mv.x                                                                     # [] Plane+

## 1. The thin lens as a collineation

At its home position a thin lens fixes its own plane and moves every other point along its line to the origin, by `home & Point`, its incidence with that plane, over the focal length; on lines it subtracts `origin & (Line ^ home)`, the origin joined with the line's crossing of the plane, over the focal length. A motor places it. The train's map on lines is the join of the images of points under its map on points: the two descriptions agree.

In [ ]:
def thin_lens(focal: np.ndarray) -> tuple[PointMap, LineMap]:
    """Thin lenses of the given focal lengths in the home plane, as collineations of points and
    as the induced maps on lines."""
    points = Point + origin * (home & Point) / focal                            # [lenses] Point- <- Point-
    lines = Line - (origin & (Line ^ home)) / focal                             # [lenses] Line- <- Line-
    return points, lines


def ball(radius: float) -> Quadric:
    """A ball of the given radius about the origin; its section with the home plane is the aperture rim."""
    coordinate_planes = stack((mv.x, mv.y, mv.z))                               # [3] Plane+
    return (coordinate_planes * (coordinate_planes & Point)).sum(-1) - mv.w * (mv.w & Point) * radius**2


def expose(
    scene: Point, subject: Point, focal: np.ndarray, placements: Motor, focus: Point, tilt: Motor, radius: float, rim: Point,
) -> tuple[PointMap, Camera, Motor, Quadric, Plane, Point]:
    """The scene seen through the lenses at their placements, with the sensor focused on `focus` and tilted: each
    scene point's aperture cone carried to the sensor side.

    The result is the lens train's collineation, the camera map, the sensor frame, the image cone of
    every scene point, the element planes [front, rear, sensor], and a fan of rays from the
    subject through the aperture rim, as its points on successive planes [planes + 1, rays].
    """
    # Place the lenses [front, rear], then compose their point maps and their line maps.
    lens_points, lens_lines = thin_lens(focal)                                  # [lenses] Point- <- Point-, Line- <- Line-
    point_maps = placements >> lens_points(placements << Point)                 # [lenses] Point- <- Point-
    line_maps = placements >> lens_lines(placements << Line)                    # [lenses] Line- <- Line-
    front, rear = line_maps[0], line_maps[1]                                    # [] Line- <- Line-, Line- <- Line-
    front_plane, rear_plane = placements[0] >> home, placements[1] >> home      # [] Plane+, Plane+
    collineation = point_maps[1](point_maps[0])                                 # [] Point- <- Point-
    train = rear(front)                                                         # [] Line- <- Line-
    pupil_ball = placements[0] >> ball(radius)(placements[0] << Point)          # [] Plane+ <- Point-

    # The focus point's image places the sensor; tilt turns it about that image.
    image = unit(collineation(focus))                                           # [] Point-
    frame = (image / origin).square_root() * tilt                               # [] Motor+
    sensor = frame >> home                                                      # [] Plane+
    camera = train(Point & Point) ^ sensor                                      # [] Point- <- (Point-, Point-)

    # Project through each scene point onto the pupil, then pull back its quadric.
    # Pull back once more through the inverse lens train to get the image cones.
    projection = (scene & Point) ^ front_plane                                  # [depths, heights, widths] Point- <- Point-
    aperture_cones = projection.adjugate()(pupil_ball(projection))              # [depths, heights, widths] Plane+ <- Point-
    inverse_collineation = collineation.inverse()                               # [] Point- <- Point-
    image_cones = inverse_collineation.adjugate()(aperture_cones(inverse_collineation))   # [depths, heights, widths] Plane+ <- Point-

    rays = subject & (placements[0] >> rim)                                     # [rays] Line-
    legs = stack([subject.broadcast_to(rays.shape), rays ^ front_plane,         # [planes + 1, rays] Point-
                  front(rays) ^ rear_plane, rear(front(rays)) ^ sensor])
    return collineation, camera, frame, image_cones, stack([front_plane, rear_plane, sensor]), legs

## 2. Wide, tele, and a tilted sensor

The scene is a grid of points at three depths, each depth drawn in its own colour on the sensor. Focused on the middle depth, wide and then tele, the middle layer is sharp dots and the near and far layers blur discs; tele magnifies and blurs more. With the sensor tilted, the plane of sharp focus tilts with it, and one side of each layer comes into focus.

In [ ]:
# The focal lengths of the front and the rear lens; the front lens sits at x == 1, and the rear lens moves to zoom.
focal_lengths = np.array([1.0, 0.6])
front_at = 1.0
# The traced rays from the subject cross the aperture at its rim's top, its centre and its bottom, in aperture radii.
rim_heights = np.array([1, 0, -1])
# The stills: the rear lens wide and tele, the focus distance, the aperture radius, and the sensor's tilt.
wide_at, tele_at = 1.4, 1.8
focus_distance = 2.2
aperture = 0.45
sensor_tilt = np.radians(25)

In [ ]:
def setting(rear_at: float, focus_at: float, tilt: float, radius: float) -> tuple[PointMap, Camera, Motor, Quadric, Plane, Point]:
    """The scene exposed with the rear lens at `rear_at`, focused on the axis at `focus_at`, the sensor tilted by
    `tilt` radians, and an aperture of the given radius."""
    placements = (mv.xw * (np.array([front_at, rear_at]) / 2)).exp()            # [lenses] Motor+
    focus = origin + mv.x.dual() * focus_at                                     # [] Point-
    turn = (mv.xy * (tilt / 2)).exp()                                           # [] Motor+
    rim = origin + mv.y.dual() * (radius * rim_heights)                         # [rays] Point-
    return expose(scenarios.scene, scenarios.subject, focal_lengths, placements, focus, turn, radius, rim)


wide = setting(wide_at, -focus_distance, 0.0, aperture)
tele = setting(tele_at, -focus_distance, 0.0, aperture)
tilted = setting(tele_at, -focus_distance, sensor_tilt, aperture)
save_figure(render.draw_stills((wide, tele, tilted)), "lens_camera");

## 3. Each blur disc, two ways

The sensor's section of a point's image cone, traced around from where the chief ray through the aperture's centre meets the sensor, lies on the conic obtained the short way: the aperture disc, a flat dual quadric, pushed through the camera's map from the pupil to the sensor for that point. A point at the focused depth images to a point, its section collapsing onto the chief-ray hit.

In [ ]:
def section(cone: Quadric, start: Point, frame: Motor, n_samples: int) -> Point:
    """The boundary of the sensor's section of a cone, traced from a point inside it along the sensor plane."""
    angles = np.linspace(0.0, 2 * np.pi, n_samples)
    turns = (mv.yz * (-angles / 2)).exp()                                       # [samples] Motor+
    sensor_directions = frame >> (turns >> mv.y.dual())                         # [samples] Point-
    # The cone's form on `start + sensor_directions * step` is quadratic in the step, with these coefficients.
    a, b, c = sensor_directions & cone(sensor_directions), sensor_directions & cone(start), start & cone(start)
    # The discriminant is negative by rounding at the cone's vertex, where the section is a single point.
    root = (b * b - a * c).clip(0.0, np.inf).square_root()
    return start + sensor_directions * ((root - b) / a)


# The samples traced around the blur section.
n_samples = 48

collineation, camera, frame, cones, planes, _ = tilted
place_front = (mv.xw * (front_at / 2)).exp()                                    # [] Motor+
sensor, centre = frame >> home, place_front >> origin                           # [] Plane+, Point-
# The aperture disc as a flat dual quadric in the front lens's plane: the directions across the axis, scaled by the
# aperture, against the origin.
pupil_directions = stack((mv.y.dual(), mv.z.dual()))                            # [2] Point-
aperture_disc = (pupil_directions * (pupil_directions & Plane)).sum(-1) * aperture**2 - origin * (origin & Plane)
pupil = place_front >> aperture_disc(place_front << Plane)                      # [] Point- <- Plane+
far_corner = scenarios.scene[2, 4, 3]                                           # [] Point-
start = (collineation(far_corner) & collineation(centre)) ^ sensor              # [] Point-
boundary = section(cones[2, 4, 3], start, frame, n_samples)                     # [samples] Point-
pupil_to_sensor = camera(far_corner)                                            # [] Point- <- Point-
blur_disc = pupil_to_sensor(pupil(pupil_to_sensor.adjugate()))                  # [] Point- <- Plane+
# The blur disc and the traced boundary in the sensor's home frame, where both lie in the plane x == 0.
home_blur_disc = (frame << blur_disc(frame >> SensorPlane)).cast(SensorPoint)   # [] Point- <- Plane+
hits = (frame << boundary).cast(SensorPoint)                                    # [samples] Point-

## 4. Zooming and refocusing

The rear lens swings to zoom while the focus moves through the layers and the aperture opens and closes; below the sensor image, the side view shows the lens planes and a fan of rays from one subject through the aperture's rim, leg by leg.

In [ ]:
# The motion: the rear lens, the focus distance and the aperture radius, each about a middle and by a swing;
# its frames, and the duration of a frame in milliseconds.
rear_middle, rear_swing = 1.6, 0.2
focus_middle, focus_swing = 2.4, 0.8
aperture_middle, aperture_swing = 0.3, 0.1
n_frames = 108
frame_duration_ms = 40

In [ ]:
def motion(n_frames: int) -> Iterator[tuple]:
    """The rear lens swings to zoom while the focus and the aperture change.

    Yields per frame the rear lens position, the focus distance, the aperture radius and the exposure.
    """
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        rear_at = rear_middle + rear_swing * np.sin(phase)
        focus_at = focus_middle - focus_swing * np.cos(phase)
        radius = aperture_middle + aperture_swing * np.sin(2 * phase)
        yield rear_at, focus_at, radius, setting(rear_at, -focus_at, 0.0, radius)


show_animation(render.animate_camera(motion(n_frames), scenarios.scene), "lens_camera", frame_duration_ms)

In [ ]:
# checks
# The image cone's vertex is the collineation's image of each scene point; the traced blur section lies
# on the pushed aperture disc; a point at the focus depth images to a point.
np.testing.assert_allclose((collineation(scenarios.scene) & cones(collineation(scenarios.scene))).kernel, 0.0, atol=1e-9)
np.testing.assert_allclose((hits & home_blur_disc.solve(hits)).kernel, 0.0, atol=1e-10)
collineation, camera, frame, cones, planes, _ = wide
start = (collineation(scenarios.scene[1, 2, 1]) & collineation(centre)) ^ (frame >> home)
np.testing.assert_allclose(unit(section(cones[1, 2, 1], start, frame, n_samples)).kernel,
                           unit(start).broadcast_to((n_samples,)).kernel, atol=1e-6)